In [1]:
# Hypothesis testing in python

A/B Testing: provides a way to check outcomes of competing scenarios and decide which way to proceed. A/B testing lets you compare scenarios to see which best achieves some goal.

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm

insurance_df = pd.read_csv('Data/insurance.csv')
print(insurance_df.head())

   age     sex     bmi  children smoker     region      charges
0   19  female  27.900         0    yes  southwest  16884.92400
1   18    male  33.770         1     no  southeast   1725.55230
2   28    male  33.000         3     no  southeast   4449.46200
3   33    male  22.705         0     no  northwest  21984.47061
4   32    male  28.880         0     no  northwest   3866.85520


In [3]:
# Calculate the proportion of late shipments, the mean cases where the late column is "Yes"
smoker_prop_samp = (insurance_df['smoker'] == 'yes').mean()

# Print the results
print('Sample proportion of smokers:', smoker_prop_samp)

Sample proportion of smokers: 0.20478325859491778


The proportion of late shipments in the sample is 0.204, or 20.4%.

Since variables have arbitrary ranges and units, we need to standardize them. For example, a hypothesis test that gave different answers if the variables were in Euros instead of US dollars would be of little value. Standardization avoids that. The z-score is a standardized measure of the difference between the sample statistic and the hypothesized statistic.

In [4]:
# Calculate the sample proportion of smokers
smoker_prop_samp = (insurance_df['smoker'] == 'yes').mean()

# Generate a bootstrap distribution of the smoker proportion
insurance_boot_distn = []
for i in range(5000):
    insurance_boot_distn.append(
        (insurance_df.sample(frac=1, replace=True)['smoker'] == 'yes').mean()
    )

In [5]:
# Hypothesize that the proportion of smokers is 20%
smoker_prop_hyp = 0.20

# Calculate the standard error
std_error = np.std(insurance_boot_distn, ddof=1)

# Find z-score of smoker_prop_samp
z_score = (smoker_prop_samp - smoker_prop_hyp) / std_error

# Print z_score
print('Z score:', z_score)

Z score: 0.4251110670382092


The z-score is a standardized measure of the difference between the sample statistic and the hypothesized statistic.

Hypothesis tests are used to determine whether the sample statistic lies in the tails of the null distribution. However, the way that the alternative hypothesis is phrased affects which tail(s) we are interested in. The tails of the distribution that are relevant depend on whether the alternative hypothesis refers to "greater than", "less than", or "differences between."

In [6]:
# Calculate the z-score of smoker_prop_samp
z_score = (smoker_prop_samp - smoker_prop_hyp) / std_error  
# Calculate the p-value
p_value = 1 - norm.cdf(z_score, loc = 0, scale = 1)
# Print the p-value
print('P value:', p_value)

P value: 0.335377855293292


**Analysis**: there's about a 33% chance of seeing a smoker proportion this far from 20% (or further) purely from random sampling, if the true rate actually is 20%.
Since 0.331 is way above the standard α = 0.05 threshold, we fail to reject the null hypothesis. Conclusion: our data doesn't provide evidence that the true smoker proportion differs from 20% — the ~20.5% we're seeing is easily explained by ordinary sampling noise.

The **p-value** is a measure of the amount of evidence to reject the null hypothesis or not. By comparing the p-value to the significance level, you can make a decision about which hypothesis to support. If the p-value is less than or equal to the significance level, you reject the null hypothesis.

In [7]:
# Calculate 95% confidence interval using quantile method
lower = np.quantile(insurance_boot_distn, 0.025)
upper = np.quantile(insurance_boot_distn, 0.975)

# Print the confidence interval
print((lower, upper))

(0.18310911808669655, 0.22795216741405083)


We fail to reject the null hypothesis. There is no statistically significant evidence that the true proportion of smokers in this population differs from 20%. The observed sample proportion (20.5%) is well within the range of values we'd expect from ordinary sampling variation around a true rate of 20%.

Type I error (False positive): rejecting the null hypothesis when in fact the null hypothesis is true

Type II error (False negative): failing to reject the null hypothesis when in fact the null hypothesis is false